## 4.3 CART 决策树(回归) - 核心逻辑&手算方法
CART决策树是一种基于二叉树结构的决策树算法，适用于分类和回归任务。在CART决策树中，特征选择的指标是基尼指数（Gini）或均方误差（MSE），具体取决于任务类型。对于分类任务，CART使用基尼指数来评估特征的纯度；<br>
对于回归任务，CART使用均方误差来评估特征的分裂效果。<br>

#### 1. CART 回归树的不纯度和目标函数
由于连续数值形的特征不像离散特征那样具有明确的类别标签，也就无法计算概率，也就无法计算基于概率的指标（如基尼指数和信息增益）。因此，在CART回归树中，我们需要使用其他指标来评估特征的分裂效果。<br>
在CART回归树中，不纯度通常使用均方误差（MSE）来衡量。均方误差是指样本的实际值与预测值之间的平均平方差。对于一个节点中的样本集合，均方误差可以通过以下公式计算：<br>
$$MSE = \frac{1}{N} \sum_{i=1}^{N} (y_i - \hat{y})^2$$
其中，$N$ 是节点中的样本数量，$y_i$ 是第 $i$ 个样本的实际值，$\hat{y}$ 是节点中样本的平均值（即预测值）。<br>
CART回归树的目标函数是通过选择特征和分裂点来最小化分裂后子节点的均方误差。对于一个特征 $A$ 的一个可能的分裂点 $s$，CART回归树的目标函数定义如下：<br>
$$\text{MSE Decrease} = MSE(D) - \left( \frac{|D_{left}|}{|D|} MSE(D_{left}) + \frac{|D_{right}|}{|D|} MSE(D_{right}) \right)$$
其中，$D$ 是当前节点的数据集，$D_{left}$ 和 $D_{right}$ 分别是根据特征 $A$ 的分裂点 $s$ 将数据集 $D$ 分成的左右子集。CART回归树选择使 MSE Decrease 最大的特征和分裂点作为当前节点的分裂条件。<br>
通过这种方式，CART回归树能够有效地选择最佳的特征和分裂点来构建回归树，从而实现对数据的回归任务。CART回归树的建树过程是一个递归的过程，通过不断选择最佳特征和分裂点来构建树结构，直到满足停止条件为止。

#### 2. CART回归树的建树过程
CART回归树的建树过程可以分为以下几个步骤：
1. **选择最佳分裂特征和分裂点**：对于当前节点的数据集，CART回归树会计算每个特征的所有可能分裂点的MSE Decrease，并选择MSE Decrease最大的特征和分裂点作为当前节点的分裂条件（阈值/子集合）。
2. **分裂数据集**：根据选择的特征和分裂点，将数据集分成两个子集：左子集和右子集。
3. **递归建树**：对左子集和右子集分别重复步骤1和步骤2，直到满足停止条件（如达到最大深度、最小样本数等）。
4. **生成叶子节点**：当满足停止条件时，当前节点成为叶子节点，叶子节点的预测值通常是该节点数据集中样本的平均值。<br>
通过以上步骤，CART回归树能够逐步构建出一棵回归树，从而实现对数据的回归任务。CART回归树的建树过程是一个递归的过程，通过不断选择最佳特征和分裂点来构建树结构，直到满足停止条件为止。

#### 3. CART回归树的手算案例

##### 3.1 数据准备
我们将使用一个简单的房价数据集来演示CART回归树算法的建树过程。数据集包含以下特征：房屋面积（square footage）、卧室数量（number of bedrooms）和房价（price）。目标变量是房价（price）。我们将使用CART回归树来预测房价。

In [1]:
import pandas as pd
# 创建数据集
data = {
    "SquareFootage": [1500, 2000, 2500, 3000, 3500, 4000, 4500, 5000, 5500, 6000],
    "Bedrooms": [3, 4, 4, 5, 5, 6, 6, 7, 7, 8],
    "Price": [300000, 400000, 500000, 600000, 700000, 800000, 900000, 1000000, 1100000, 1200000]
}
df = pd.DataFrame(data)
df

,SquareFootage,Bedrooms,Price
0,1500,3,300000
1,2000,4,400000
2,2500,4,500000
3,3000,5,600000
4,3500,5,700000
5,4000,6,800000
6,4500,6,900000
7,5000,7,1000000
8,5500,7,1100000
9,6000,8,1200000


##### 3.2 计算父节点的MSE

In [5]:
# 计算父节点的均值
mean_price = df["Price"].mean()
# 计算父节点的MSE
MSE_root = ((df["Price"] - mean_price) ** 2).mean()
# 从np数据类型转换为Python内置数据类型，方便后续的计算
MSE_root = float(MSE_root)
MSE_root

82500000000.0

##### 3.3 计算子节点的MSE和MSE Decrease

###### 3.3.1 候选特征 - SquareFootage <br>
对于阈值选择，我们可以选择特征值的中点作为候选分裂点，比如：
- 我们可以选择以下候选分裂点：1750, 2250, 2750, 3250, 3750, 4250, 4750, 5250, 5750
- 分别1+n组合之后，计算每个候选分裂点的子节点的MSE和MSE Decrease，并选择MSE Decrease最大的阈值作为当前节点的分裂条件。<br>
但是对于手算来说，计算所有候选分裂点的MSE Decrease会比较繁琐，因此我们可以选择一个候选分裂点来演示计算过程。我们选择分裂点为3250来计算子节点的MSE和MSE Decrease。

In [8]:
# 计算分裂点
squareFootage_split_point = df["SquareFootage"].mean() # 3250
# 根据分裂点将数据集分成左右子集
squareFootage_left_split = df[df["SquareFootage"] <= split_point]
squareFootage_right_split = df[df["SquareFootage"] > split_point]
# 计算左子集的均值和MSE
squareFootage_mean_left = squareFootage_left_split["Price"].mean()
squareFootage_MSE_left = ((squareFootage_left_split["Price"] - squareFootage_mean_left) ** 2).mean()
# 计算右子集的均值和MSE
squareFootage_mean_right = squareFootage_right_split["Price"].mean()
squareFootage_MSE_right = ((right_split["Price"] - mean_right) ** 2).mean()
# 从np数据类型转换为Python内置数据类型，方便后续的计算
squareFootage_MSE_left = float(squareFootage_MSE_left)
squareFootage_MSE_right = float(squareFootage_MSE_right)
# 计算加权平均MSE
squareFootage_MSE_split = (len(squareFootage_left_split) / len(df)) * squareFootage_MSE_left + (len(squareFootage_right_split) / len(df)) * squareFootage_MSE_right
# 计算MSE Decrease
squareFootage_MSE_Decrease = MSE_root - squareFootage_MSE_split
squareFootage_MSE_Decrease

62500000000.0

###### 3.3.2 候选特征 - Bedrooms <br>
同样的，对于阈值的选择，作为手算演示，我们选择分裂点为5.5来计算子节点的MSE和MSE Decrease。

In [9]:
# 计算分裂点
bedrooms_split_point = df["Bedrooms"].mean() # 5.5
# 根据分裂点将数据集分成左右子集
bedrooms_left_split = df[df["Bedrooms"] <= bedrooms_split_point]
bedrooms_right_split = df[df["Bedrooms"] > bedrooms_split_point]
# 计算左子集的均值和MSE
bedrooms_mean_left = bedrooms_left_split["Price"].mean()
bedrooms_MSE_left = ((bedrooms_left_split["Price"] - bedrooms_mean_left) ** 2).mean()
# 计算右子集的均值和MSE
bedrooms_mean_right = bedrooms_right_split["Price"].mean()
bedrooms_MSE_right = ((bedrooms_right_split["Price"] - bedrooms_mean_right) ** 2).mean()
# 从np数据类型转换为Python内置数据类型，方便后续的计算
bedrooms_MSE_left = float(bedrooms_MSE_left)
bedrooms_MSE_right = float(bedrooms_MSE_right)
# 计算加权平均MSE
bedrooms_MSE_split = (len(bedrooms_left_split) / len(df)) * bedrooms_MSE_left + (len(bedrooms_right_split) / len(df)) * bedrooms_MSE_right
# 计算MSE Decrease
bedrooms_MSE_Decrease = MSE_root - bedrooms_MSE_split
bedrooms_MSE_Decrease

62500000000.0

###### 3.3.3 选择最优特征和分裂点 <br>
通过比较每个候选特征和分裂点的MSE Decrease，我们可以选择MSE Decrease最大的特征和分裂点作为当前节点的分裂条件。<br>
对于本例，两个MSE Decrease一样，我们可以选择任意一个特征和分裂点作为当前节点的分裂条件。假设我们选择特征SquareFootage和分裂点3250作为当前节点的分裂条件，那么我们就可以根据这个分裂条件将数据集分成左右子集，并继续对每个子集进行递归建树，直到满足停止条件为止。

#### 4. 分裂后得到的结果
通过对根节点进行分裂，原始数据被拆分为两个子节点，分别对应特征SquareFootage的取值范围：小于等于3250和大于3250。 <br>
- 对于SquareFootage <= 3250的子节点，我们需要继续计算特征Bedrooms的MSE Decrease，并选择MSE Decrease最大的特征和分裂点进行划分。 <br>
- 对于SquareFootage > 3250的子节点，我们需要继续计算特征Bedrooms的MSE Decrease，并选择MSE Decrease最大的特征和分裂点进行划分。 <br>
通过递归地选择最优特征和分裂点进行划分，我们能够逐步构建出一个树形结构，从而实现对数据的回归预测。 <br>
通过以上步骤，我们可以构建出一个完整的CART回归树，从而实现对数据的回归预测。